# 🔄 Notebook 4: State Synchronization

Your database says 'pending' but S3 has the file. Or vice versa. Let's fix that.

## Learning Objectives

By the end of this notebook, you'll understand:
- The dual-state problem
- Event-driven state updates
- Reconciliation patterns
- Handling edge cases

In [1]:
import boto3
from botocore.config import Config
import psycopg2
import uuid
from datetime import datetime, timedelta
from typing import Optional

s3 = boto3.client(
    's3',
    endpoint_url='http://localhost:9000',
    aws_access_key_id='minioadmin',
    aws_secret_access_key='minioadmin',
    config=Config(signature_version='s3v4'),
    region_name='us-east-1'
)

conn = psycopg2.connect(
    host="localhost", port=5432,
    database="largeblobs", user="postgres", password="postgres"
)
conn.autocommit = True

BUCKET = 'uploads'

print("✅ Connected to MinIO and PostgreSQL!")
print("📊 MinIO Console: http://localhost:9001")
print("📊 Adminer: http://localhost:8080")

✅ Connected to MinIO and PostgreSQL!
📊 MinIO Console: http://localhost:9001
📊 Adminer: http://localhost:8080


## 🔄 The Dual-State Problem

In [2]:
print("🔄 The Dual-State Problem")
print("=" * 60)
print("""
With direct uploads, you have TWO sources of truth:

┌─────────────────────────────────────────────────────────────┐
│                      DATABASE                                │
│  file_id  │ filename    │ storage_key      │ status         │
│  abc-123  │ video.mp4   │ user1/video.mp4  │ 'pending'  ?   │
└─────────────────────────────────────────────────────────────┘
                           vs
┌─────────────────────────────────────────────────────────────┐
│                      BLOB STORAGE                            │
│  Key: user1/video.mp4                                        │
│  Status: EXISTS (500MB uploaded!)                            │
└─────────────────────────────────────────────────────────────┘

PROBLEM SCENARIOS:
─────────────────────────────────────────────────────────────

1. DB says 'pending', file exists in S3
   → Client uploaded but didn't notify server

2. DB says 'completed', file missing in S3
   → File deleted? Never uploaded? Bug?

3. DB says 'uploading', upload abandoned
   → Incomplete multipart taking up space

4. File exists in S3, no DB record
   → Orphaned file, nobody knows about it
""")

🔄 The Dual-State Problem

With direct uploads, you have TWO sources of truth:

┌─────────────────────────────────────────────────────────────┐
│                      DATABASE                                │
│  file_id  │ filename    │ storage_key      │ status         │
│  abc-123  │ video.mp4   │ user1/video.mp4  │ 'pending'  ?   │
└─────────────────────────────────────────────────────────────┘
                           vs
┌─────────────────────────────────────────────────────────────┐
│                      BLOB STORAGE                            │
│  Key: user1/video.mp4                                        │
│  Status: EXISTS (500MB uploaded!)                            │
└─────────────────────────────────────────────────────────────┘

PROBLEM SCENARIOS:
─────────────────────────────────────────────────────────────

1. DB says 'pending', file exists in S3
   → Client uploaded but didn't notify server

2. DB says 'completed', file missing in S3
   → File deleted? Never uploaded?

## 📝 Proper Upload Flow

In [3]:
class FileUploadService:
    def __init__(self, conn, s3_client, bucket: str):
        self.conn = conn
        self.s3 = s3_client
        self.bucket = bucket
    
    def request_upload(self, user_id: str, filename: str, size_bytes: int) -> dict:
        file_id = str(uuid.uuid4())
        storage_key = f"{user_id}/{file_id}/{filename}"
        
        cursor = self.conn.cursor()
        cursor.execute("""
            INSERT INTO files (id, user_id, filename, size_bytes, storage_key, status)
            VALUES (%s, %s, %s, %s, %s, 'pending')
        """, (file_id, user_id, filename, size_bytes, storage_key))
        cursor.close()
        
        presigned_url = self.s3.generate_presigned_url(
            'put_object',
            Params={'Bucket': self.bucket, 'Key': storage_key},
            ExpiresIn=3600
        )
        
        return {
            'file_id': file_id,
            'upload_url': presigned_url,
            'storage_key': storage_key
        }
    
    def confirm_upload(self, file_id: str) -> bool:
        cursor = self.conn.cursor()
        cursor.execute("SELECT storage_key FROM files WHERE id = %s", (file_id,))
        row = cursor.fetchone()
        
        if not row:
            cursor.close()
            return False
        
        storage_key = row[0]
        
        try:
            head = self.s3.head_object(Bucket=self.bucket, Key=storage_key)
            actual_size = head['ContentLength']
            
            cursor.execute("""
                UPDATE files 
                SET status = 'completed', size_bytes = %s, updated_at = NOW()
                WHERE id = %s
            """, (actual_size, file_id))
            cursor.close()
            return True
            
        except Exception:
            cursor.close()
            return False
    
    def get_file_status(self, file_id: str) -> dict:
        cursor = self.conn.cursor()
        cursor.execute("""
            SELECT filename, storage_key, status, size_bytes 
            FROM files WHERE id = %s
        """, (file_id,))
        row = cursor.fetchone()
        cursor.close()
        
        if not row:
            return None
        
        return {
            'filename': row[0],
            'storage_key': row[1],
            'status': row[2],
            'size_bytes': row[3]
        }

service = FileUploadService(conn, s3, BUCKET)
print("✅ FileUploadService ready!")

✅ FileUploadService ready!


In [4]:
print("📤 Complete Upload Flow Demo")
print("=" * 60)

user_id = '11111111-1111-1111-1111-111111111111'

print("\n1️⃣ Request upload URL...")
upload_info = service.request_upload(user_id, 'demo-video.mp4', 1024000)
print(f"   File ID: {upload_info['file_id']}")
print(f"   Status in DB: pending")

print("\n2️⃣ Upload directly to storage...")
import requests
file_content = b"This is demo video content!" * 1000
response = requests.put(upload_info['upload_url'], data=file_content)
print(f"   Upload status: {response.status_code}")

print("\n3️⃣ Confirm upload (verify file exists)...")
confirmed = service.confirm_upload(upload_info['file_id'])
print(f"   Confirmed: {'✅ Yes' if confirmed else '❌ No'}")

print("\n4️⃣ Check final status...")
status = service.get_file_status(upload_info['file_id'])
print(f"   Status: {status['status']}")
print(f"   Size: {status['size_bytes']} bytes")

📤 Complete Upload Flow Demo

1️⃣ Request upload URL...
   File ID: 6e9d90fe-82f6-4307-97b9-e2344bb30601
   Status in DB: pending

2️⃣ Upload directly to storage...
   Upload status: 200

3️⃣ Confirm upload (verify file exists)...
   Confirmed: ✅ Yes

4️⃣ Check final status...
   Status: completed
   Size: 27000 bytes


## 🔍 Reconciliation Pattern

In [5]:
print("🔍 Reconciliation Pattern")
print("=" * 60)
print("""
Even with proper flow, things can go wrong:
• Client crashes after upload, before confirm
• Network fails during confirm call
• Event notification gets lost

SOLUTION: Periodic reconciliation job
─────────────────────────────────────────────────────────────

Every 5 minutes:

1. Find 'pending' files older than 1 hour
2. Check if file exists in S3
3. If exists → mark 'completed'
4. If missing → mark 'failed' (and cleanup)
""")

🔍 Reconciliation Pattern

Even with proper flow, things can go wrong:
• Client crashes after upload, before confirm
• Network fails during confirm call
• Event notification gets lost

SOLUTION: Periodic reconciliation job
─────────────────────────────────────────────────────────────

Every 5 minutes:

1. Find 'pending' files older than 1 hour
2. Check if file exists in S3
3. If exists → mark 'completed'
4. If missing → mark 'failed' (and cleanup)



In [6]:
class ReconciliationJob:
    def __init__(self, conn, s3_client, bucket: str):
        self.conn = conn
        self.s3 = s3_client
        self.bucket = bucket
    
    def reconcile_pending(self, older_than_minutes: int = 60):
        cursor = self.conn.cursor()
        
        cursor.execute("""
            SELECT id, storage_key, filename 
            FROM files 
            WHERE status = 'pending' 
            AND created_at < NOW() - INTERVAL '%s minutes'
        """, (older_than_minutes,))
        
        pending_files = cursor.fetchall()
        results = {'confirmed': 0, 'failed': 0, 'total': len(pending_files)}
        
        for file_id, storage_key, filename in pending_files:
            try:
                head = self.s3.head_object(Bucket=self.bucket, Key=storage_key)
                cursor.execute("""
                    UPDATE files 
                    SET status = 'completed', size_bytes = %s, updated_at = NOW()
                    WHERE id = %s
                """, (head['ContentLength'], file_id))
                results['confirmed'] += 1
                print(f"   ✅ {filename}: Confirmed (was uploaded)")
                
            except self.s3.exceptions.ClientError:
                cursor.execute("""
                    UPDATE files SET status = 'failed', updated_at = NOW()
                    WHERE id = %s
                """, (file_id,))
                results['failed'] += 1
                print(f"   ❌ {filename}: Failed (never uploaded)")
        
        cursor.close()
        return results
    
    def find_orphaned_files(self) -> list:
        cursor = self.conn.cursor()
        cursor.execute("SELECT storage_key FROM files WHERE status = 'completed'")
        known_keys = {row[0] for row in cursor.fetchall()}
        cursor.close()
        
        orphaned = []
        paginator = self.s3.get_paginator('list_objects_v2')
        
        for page in paginator.paginate(Bucket=self.bucket):
            for obj in page.get('Contents', []):
                if obj['Key'] not in known_keys:
                    orphaned.append(obj['Key'])
        
        return orphaned

reconciler = ReconciliationJob(conn, s3, BUCKET)
print("✅ ReconciliationJob ready!")

✅ ReconciliationJob ready!


In [7]:
print("🔍 Reconciliation Demo")
print("=" * 60)

print("\n📝 Creating test scenarios...")

cursor = conn.cursor()
old_file_id = str(uuid.uuid4())
cursor.execute("""
    INSERT INTO files (id, user_id, filename, storage_key, status, created_at)
    VALUES (%s, %s, 'old-pending.mp4', 'test/old-pending.mp4', 'pending', NOW() - INTERVAL '2 hours')
""", (old_file_id, user_id))

s3.put_object(Bucket=BUCKET, Key='test/old-pending.mp4', Body=b'test content')
print("   Created: pending file that was actually uploaded")

abandoned_id = str(uuid.uuid4())
cursor.execute("""
    INSERT INTO files (id, user_id, filename, storage_key, status, created_at)
    VALUES (%s, %s, 'abandoned.mp4', 'test/abandoned.mp4', 'pending', NOW() - INTERVAL '3 hours')
""", (abandoned_id, user_id))
print("   Created: pending file that was never uploaded")

cursor.close()

print("\n🔄 Running reconciliation (for files > 1 hour old)...")
results = reconciler.reconcile_pending(older_than_minutes=60)

print(f"\n📊 Results:")
print(f"   Total checked: {results['total']}")
print(f"   Confirmed: {results['confirmed']}")
print(f"   Failed: {results['failed']}")

🔍 Reconciliation Demo

📝 Creating test scenarios...
   Created: pending file that was actually uploaded
   Created: pending file that was never uploaded

🔄 Running reconciliation (for files > 1 hour old)...
   ✅ old-pending.mp4: Confirmed (was uploaded)
   ❌ abandoned.mp4: Failed (never uploaded)

📊 Results:
   Total checked: 2
   Confirmed: 1
   Failed: 1


## 🧪 Quick Quiz

1. **Why can't you trust the client's "upload complete" notification?**

2. **What's an orphaned file and how does it happen?**

3. **Why run reconciliation periodically instead of immediately?**

In [8]:
print("📝 Quiz Answers")
print("=" * 50)
print()
print("1. Can't trust client notification:")
print("   - Client could lie (security)")
print("   - Client could crash before notifying")
print("   - Network could fail during notify")
print("   - Always verify with storage!")
print()
print("2. Orphaned files:")
print("   - File exists in S3, no DB record")
print("   - Causes: failed DB insert, bug, manual upload")
print("   - Problem: costs money, no one knows it's there")
print()
print("3. Why periodic reconciliation:")
print("   - Give time for normal flow to complete")
print("   - Reduce unnecessary S3 API calls")
print("   - Batch multiple checks efficiently")

📝 Quiz Answers

1. Can't trust client notification:
   - Client could lie (security)
   - Client could crash before notifying
   - Network could fail during notify
   - Always verify with storage!

2. Orphaned files:
   - File exists in S3, no DB record
   - Causes: failed DB insert, bug, manual upload
   - Problem: costs money, no one knows it's there

3. Why periodic reconciliation:
   - Give time for normal flow to complete
   - Reduce unnecessary S3 API calls
   - Batch multiple checks efficiently


## 📚 Summary

### Key Takeaways

1. **Dual-state problem** - DB and storage can disagree
2. **Verify uploads** - Check storage before marking complete
3. **Reconciliation** - Periodic job to catch edge cases
4. **Find orphans** - Clean up files with no DB record
5. **Never trust clients** - Always verify server-side

### Next Up

In **Notebook 5**, we'll learn about download optimization:
- Range requests for resumable downloads
- Parallel chunk downloads
- CDN caching strategies